# `q_diagonalize()`

`nematics3d.q_diagonalize()` extracts scalar order and a principal director from a symmetric traceless $Q$-tensor field. When requested, it also returns the complete eigensystem and biaxial order.

The implementation has two analytic paths:

- the default principal-only path computes only the largest eigenvalue and its director;
- the complete path computes an isolated eigenpair, then solves the remaining symmetric $2\times2$ problem in its orthogonal plane.

Pure uniaxiality is therefore a normal input state, not an eigensolver failure that needs a fallback.

## Setup

The examples use only NumPy and Nematics3D.

In [1]:
import numpy as np
import nematics3d as n3d

## Minimal example

For a positive uniaxial state,

$$Q=S\left(\mathbf n\mathbf n-\frac{I}{3}\right).$$

The default mode returns the scalar order $S=3\lambda_{\max}/2$ and the unit eigenvector belonging to $\lambda_{\max}$.

In [2]:
S_input = 0.75
n_input = np.array([1.0, 1.0, 1.0])
n_input /= np.linalg.norm(n_input)
Q = S_input * (np.outer(n_input, n_input) - np.eye(3) / 3.0)

result = n3d.q_diagonalize(Q, log_mode="none")
print("S =", result.S)
print("n =", result.n)
print("axis overlap =", abs(np.dot(result.n, n_input)))

S = 0.7500000000000002
n = [0.57735027 0.57735027 0.57735027]
axis overlap = 1.0


The sign of an eigenvector is arbitrary. Compare nematic axes with $|\mathbf n_1\cdot\mathbf n_2|$, not by requiring componentwise equality.

## Inputs and outputs

The public signature is:

```python
q_diagonalize(qtensor, *, is_biaxial=False, is_right_handed=False)
```

`qtensor` must be a non-empty floating-point array with trailing shape `(..., 5)` or `(..., 3, 3)`. Full matrices are validated as symmetric and traceless. Compact tensors use

```text
(Q_xx, Q_xy, Q_xz, Q_yy, Q_yz)
```

The result fields are:

- `S`: $3\lambda_{\max}/2$;
- `n`: the corresponding unit director;
- `isotropic_indices`: coordinates assigned the deterministic isotropic convention;
- `uniaxial_indices`: positive-uniaxial coordinates classified in complete mode;
- `eigenvalues`, `eigenvectors`, `biaxial_order`: present only in complete mode.

### Compact five-component input

In [3]:
Q5 = np.array([Q[0, 0], Q[0, 1], Q[0, 2], Q[1, 1], Q[1, 2]])
compact_result = n3d.q_diagonalize(Q5, log_mode="none")
print("same S:", np.allclose(compact_result.S, result.S))
print("same axis:", abs(np.dot(compact_result.n, result.n)))

same S: True
same axis: 0.9999999999999999


### A field or batch of tensors

All leading dimensions are preserved.

In [4]:
Q_batch = np.stack([Q, 0.5 * Q])
batch_result = n3d.q_diagonalize(Q_batch, log_mode="none")
print("input:", Q_batch.shape)
print("S:", batch_result.S.shape, batch_result.S)
print("n:", batch_result.n.shape)

input: (2, 3, 3)
S: (2,) [0.75  0.375]
n: (2, 3)


## Complete eigensystem and biaxial order

Set `is_biaxial=True` to return descending eigenvalues and matching eigenvector columns. Nematics3D defines

$$b=\frac32|\lambda_1-\lambda_2|,$$

where $\lambda_0\geq\lambda_1\geq\lambda_2$.

In [5]:
axes, _ = np.linalg.qr(np.random.default_rng(7).normal(size=(3, 3)))
expected_values = np.array([0.6, -0.1, -0.5])
Q_biaxial = axes @ np.diag(expected_values) @ axes.T

complete = n3d.q_diagonalize(
    Q_biaxial,
    is_biaxial=True,
    is_right_handed=True,
    log_mode="none",
)
Q_reconstructed = np.einsum(
    "...ik,...k,...jk->...ij",
    complete.eigenvectors,
    complete.eigenvalues,
    complete.eigenvectors,
)
print("eigenvalues =", complete.eigenvalues)
print("S =", complete.S)
print("biaxial order =", complete.biaxial_order)
print("right-handed determinant =", np.linalg.det(complete.eigenvectors))
print("reconstruction error =", np.max(np.abs(Q_reconstructed - Q_biaxial)))

eigenvalues = [ 0.6 -0.1 -0.5]
S = 0.8999999999999999
biaxial order = 0.6000000000000001
right-handed determinant = 1.0
reconstruction error = 2.220446049250313e-16


## Uniaxiality is not a numerical exception

In a positive uniaxial tensor, the two lower eigenvalues are equal. Their individual eigenvectors are not unique, but their two-dimensional eigenspace is. The complete solver constructs an orthonormal basis inside that plane, so it does not need `numpy.linalg.eigh()` or an axis-specific fallback.

`uniaxial_indices` remains useful as physical metadata and to set roundoff-level biaxial order exactly to zero. It is no longer needed to rescue the eigensolver.

In [6]:
Q_x_uniaxial = 1.0 * (
    np.outer([1.0, 0.0, 0.0], [1.0, 0.0, 0.0]) - np.eye(3) / 3.0
)
x_result = n3d.q_diagonalize(
    Q_x_uniaxial[None, ...],
    is_biaxial=True,
    is_right_handed=True,
    log_mode="none",
)
frame_product = np.swapaxes(x_result.eigenvectors, -1, -2) @ x_result.eigenvectors
print("S =", x_result.S)
print("n =", x_result.n)
print("uniaxial indices =", x_result.uniaxial_indices)
print("orthonormal frame =", np.allclose(frame_product, np.eye(3)))

S = [1.]
n = [[1. 0. 0.]]
uniaxial indices = [(0,)]
orthonormal frame = True


The perfectly $x$-aligned case is deliberately shown because the retired fixed-cofactor formula could vanish there. The current strongest-adjugate-row selection treats $x$, $y$, $z$, and generic orientations uniformly.

## The isotropic case still needs a convention

At `Q = 0`, all directions are eigenvectors, so no physical director exists. The function returns `S = 0`, assigns `[1, 0, 0]` as a deterministic placeholder, and records the coordinate in `isotropic_indices`.

In [7]:
isotropic = n3d.q_diagonalize(np.zeros((3, 3)), log_mode="none")
print("S =", isotropic.S)
print("placeholder n =", isotropic.n)
print("isotropic indices =", isotropic.isotropic_indices)

S = 0.0
placeholder n = [1. 0. 0.]
isotropic indices = [()]


## Current limitation: negative-$S$ uniaxial states

The public `S` and `n` convention still selects the largest eigenpair. For an oblate state with $S=-1/2$, the spectrum is $(1/6,1/6,-1/3)$. The physical symmetry axis belongs to the unique smallest eigenvalue, so the current function reports $S=1/4$ and an arbitrary axis in the repeated upper eigenspace.

This is a definition/classification limitation, not an eigendecomposition instability. Supporting negative $S$ requires detecting an upper repeated pair and selecting the unique smallest eigenpair.

## Performance benchmarks

The following benchmarks use `example/data/Q_example_workflow.npy` and run in fresh subprocesses. Before NumPy, NumExpr, or Nematics3D is imported, all common numerical-library thread variables are set to `1`. The subprocess is additionally pinned to one logical CPU with `psutil`, when supported.

Times are the median of three measured runs after warm-up. “Peak MiB” is the incremental peak tracked by `tracemalloc`; it is not total process RSS. Input preparation and correctness checks occur outside the timed region.

In [8]:
import json
import os
import platform
import subprocess
import sys
import textwrap
from pathlib import Path


def find_repo_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "example/data/Q_example_workflow.npy").exists():
            return candidate
    raise FileNotFoundError("Run this tutorial from inside the Nematics3D repository.")


repo_root = find_repo_root()
q5_path = repo_root / "example/data/Q_example_workflow.npy"

benchmark_program = textwrap.dedent(
    r"""
    import gc
    import json
    import os
    import platform
    import statistics
    import sys
    import time
    import tracemalloc

    import numexpr as ne
    import numpy as np
    import psutil

    def cpu_model():
        if platform.system() == "Windows":
            try:
                import winreg

                key_path = r"HARDWARE\DESCRIPTION\System\CentralProcessor\0"
                with winreg.OpenKey(winreg.HKEY_LOCAL_MACHINE, key_path) as key:
                    return winreg.QueryValueEx(key, "ProcessorNameString")[0].strip()
            except OSError:
                pass
        return platform.processor() or platform.machine()

    process = psutil.Process()
    affinity = process.cpu_affinity() if hasattr(process, "cpu_affinity") else []
    if affinity:
        process.cpu_affinity([affinity[0]])
        affinity = process.cpu_affinity()
    ne.set_num_threads(1)

    sys.path.insert(0, sys.argv[2])
    from nematics3d import q_diagonalize
    from nematics3d.datatypes import as_qfield9

    mode = sys.argv[1]
    q5 = np.load(sys.argv[3])
    q9 = np.asarray(
        as_qfield9(q5, is_strict_3d_field=False), dtype=np.float64
    )
    # Remove float32 expansion roundoff before strict float64 validation.
    q9[..., 2, 2] = -q9[..., 0, 0] - q9[..., 1, 1]

    def measure(operation, repeats=3):
        times = []
        peaks = []
        result = None
        for _ in range(repeats):
            gc.collect()
            tracemalloc.start()
            start = time.perf_counter()
            result = operation()
            times.append(time.perf_counter() - start)
            peaks.append(tracemalloc.get_traced_memory()[1] / 1024**2)
            tracemalloc.stop()
        return result, statistics.median(times), max(peaks)

    q_diagonalize(q9.reshape(-1, 3, 3)[:1000], is_biaxial=(mode == "biaxial"), log_mode="none")

    blas = np.show_config(mode="dicts").get("Build Dependencies", {}).get(
        "blas", {}
    )
    metadata = {
        "mode": mode,
        "os": platform.platform(),
        "cpu": cpu_model(),
        "physical_cores": psutil.cpu_count(logical=False),
        "logical_cores": psutil.cpu_count(logical=True),
        "total_memory_gib": psutil.virtual_memory().total / 1024**3,
        "python": platform.python_version(),
        "numpy": np.__version__,
        "numexpr": ne.__version__,
        "numexpr_threads": ne.get_num_threads(),
        "blas": (blas.get("name", "unavailable") + " " + blas.get("version", "")).strip(),
        "cpu_affinity": affinity,
        "field_shape": q5.shape,
        "field_dtype": str(q5.dtype),
        "tensor_count": q5.size // 5,
    }

    if mode == "principal":
        result, seconds, peak_mib = measure(
            lambda: q_diagonalize(q9, log_mode="none")
        )
        residual = np.max(
            np.abs(
                np.einsum("...ij,...j->...i", q9, result.n)
                - (result.S / 1.5)[..., None] * result.n
            )
        )
        metadata.update(
            {
                "n3d_seconds": seconds,
                "n3d_peak_mib": peak_mib,
                "max_principal_residual": float(residual),
            }
        )
    else:
        result, n3d_seconds, n3d_peak_mib = measure(
            lambda: q_diagonalize(q9, is_biaxial=True, log_mode="none")
        )
        numpy_result, numpy_seconds, numpy_peak_mib = measure(
            lambda: np.linalg.eigh(q9)
        )
        numpy_values, numpy_vectors = numpy_result
        numpy_values = numpy_values[..., ::-1]
        numpy_vectors = numpy_vectors[..., :, ::-1]
        overlaps = np.abs(
            np.einsum("...ij,...ij->...j", result.eigenvectors, numpy_vectors)
        )
        reconstructed = np.einsum(
            "...ik,...k,...jk->...ij",
            result.eigenvectors,
            result.eigenvalues,
            result.eigenvectors,
        )
        metadata.update(
            {
                "n3d_seconds": n3d_seconds,
                "n3d_peak_mib": n3d_peak_mib,
                "numpy_seconds": numpy_seconds,
                "numpy_peak_mib": numpy_peak_mib,
                "max_eigenvalue_difference": float(
                    np.max(np.abs(result.eigenvalues - numpy_values))
                ),
                "minimum_axis_overlap": float(np.min(overlaps)),
                "max_reconstruction_error": float(
                    np.max(np.abs(reconstructed - q9))
                ),
            }
        )
    print(json.dumps(metadata))
    """
)


def run_single_core_benchmark(mode):
    environment = os.environ.copy()
    environment.update(
        {
            "NUMEXPR_MAX_THREADS": "1",
            "NUMEXPR_NUM_THREADS": "1",
            "OMP_NUM_THREADS": "1",
            "OPENBLAS_NUM_THREADS": "1",
            "MKL_NUM_THREADS": "1",
            "BLIS_NUM_THREADS": "1",
            "VECLIB_MAXIMUM_THREADS": "1",
        }
    )
    completed = subprocess.run(
        [
            sys.executable,
            "-c",
            benchmark_program,
            mode,
            str(repo_root / "src"),
            str(q5_path),
        ],
        check=False,
        capture_output=True,
        text=True,
        env=environment,
    )
    if completed.returncode:
        raise RuntimeError(
            "benchmark subprocess failed:\n" + completed.stdout + completed.stderr
        )
    return json.loads(completed.stdout.strip())


def print_hardware(result):
    print("OS:", result["os"])
    print("CPU:", result["cpu"])
    print(
        "CPU cores:",
        result["physical_cores"],
        "physical,",
        result["logical_cores"],
        "logical",
    )
    print("benchmark CPU affinity:", result["cpu_affinity"])
    print("system memory: %.1f GiB" % result["total_memory_gib"])
    print(
        "software: Python %s, NumPy %s, NumExpr %s"
        % (result["python"], result["numpy"], result["numexpr"])
    )
    print("NumExpr threads:", result["numexpr_threads"])
    print("BLAS:", result["blas"])
    print("GPU: not used")
    print(
        "field:",
        tuple(result["field_shape"]),
        result["field_dtype"],
        "(%s tensors)" % f'{result["tensor_count"]:,}',
    )

### Benchmark 1: principal-only path, strictly one CPU core

This benchmark measures the optimized `is_biaxial=False` path. It returns only `S` and `n`; it does not allocate or solve the orthogonal two-dimensional eigenspace.

In [9]:
principal_benchmark = run_single_core_benchmark("principal")
print_hardware(principal_benchmark)
print(
    "q_diagonalize principal-only: %.3f s, %.1f MiB peak"
    % (
        principal_benchmark["n3d_seconds"],
        principal_benchmark["n3d_peak_mib"],
    )
)
print(
    "maximum principal eigenpair residual: %.3e"
    % principal_benchmark["max_principal_residual"]
)

OS: Windows-11-10.0.26200-SP0
CPU: 13th Gen Intel(R) Core(TM) i7-13700K
CPU cores: 16 physical, 24 logical
benchmark CPU affinity: [0]
system memory: 79.7 GiB
software: Python 3.12.11, NumPy 2.3.2, NumExpr 2.14.2
NumExpr threads: 1
BLAS: blas 3.9.0
GPU: not used
field: (200, 100, 100, 5) float32 (2,000,000 tensors)
q_diagonalize principal-only: 0.574 s, 267.0 MiB peak
maximum principal eigenpair residual: 1.707e-15


### Benchmark 2: complete eigensystem versus `numpy.linalg.eigh`, strictly one CPU core

Both functions receive the same prepared `float64` full-Q array. Nematics3D returns eigenvalues in descending order, while NumPy returns them in ascending order, so the NumPy result is reversed before comparison.

In [10]:
biaxial_benchmark = run_single_core_benchmark("biaxial")
print_hardware(biaxial_benchmark)
print(
    "q_diagonalize complete: %.3f s, %.1f MiB peak"
    % (
        biaxial_benchmark["n3d_seconds"],
        biaxial_benchmark["n3d_peak_mib"],
    )
)
print(
    "numpy.linalg.eigh: %.3f s, %.1f MiB peak"
    % (
        biaxial_benchmark["numpy_seconds"],
        biaxial_benchmark["numpy_peak_mib"],
    )
)
print(
    "maximum eigenvalue difference: %.3e"
    % biaxial_benchmark["max_eigenvalue_difference"]
)
print(
    "minimum absolute eigenvector overlap: %.12f"
    % biaxial_benchmark["minimum_axis_overlap"]
)
print(
    "maximum Q reconstruction error: %.3e"
    % biaxial_benchmark["max_reconstruction_error"]
)

OS: Windows-11-10.0.26200-SP0
CPU: 13th Gen Intel(R) Core(TM) i7-13700K
CPU cores: 16 physical, 24 logical
benchmark CPU affinity: [0]
system memory: 79.7 GiB
software: Python 3.12.11, NumPy 2.3.2, NumExpr 2.14.2
NumExpr threads: 1
BLAS: blas 3.9.0
GPU: not used
field: (200, 100, 100, 5) float32 (2,000,000 tensors)
q_diagonalize complete: 1.093 s, 360.5 MiB peak
numpy.linalg.eigh: 2.721 s, 183.1 MiB peak
maximum eigenvalue difference: 7.772e-16
minimum absolute eigenvector overlap: 1.000000000000
maximum Q reconstruction error: 4.441e-16


Benchmark results depend on CPU frequency, memory bandwidth, operating-system load, NumPy build, and NumExpr version. Keep the hardware and software report with any quoted timing; do not compare numbers collected under different thread limits as though they were equivalent.